# 生成結果を 1 文字ずつ表示する（ストリーミング）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm/llm_streaming_local.ipynb)

NLP クックブック 14-02 ／ 大規模言語モデル (LLM)

生成が終わるまで待たずに、出力されたそばから表示したい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
model = AutoModelForCausalLM.from_pretrained(name).eval()

messages = [{"role": "user", "content": "自然言語処理とは何か、3 文で説明してください。"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)   # 標準出力に逐次表示
_ = model.generate(**inputs, max_new_tokens=120, do_sample=False, streamer=streamer, pad_token_id=tokenizer.eos_token_id)

自然言語処理（NLP）とは、自然言語をコンピュータが理解、生成、分析できるようにする技術分野です。これは、テキストデータから意味を抽出し、感情分析、機械翻訳、質問応答、言語モデルなど、さまざまなタスクを実行することで、情報の取得やコミュニケーションを円滑にする役割を果たします。


In [2]:
# 自分のコードで受け取りたい場合（Web アプリに流すなど）: 別スレッドで生成し、イテレータから読む
from threading import Thread
from transformers import TextIteratorStreamer

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
thread = Thread(target=model.generate, kwargs=dict(**inputs, max_new_tokens=60, do_sample=False, streamer=streamer, pad_token_id=tokenizer.eos_token_id))
thread.start()
chunks = [chunk for chunk in streamer]
thread.join()
# 断片は、区切りのよいところ（空白・改行・漢字）までたまってから届き、たまっている間は空の文字列 "" が届く。表示では空のものを除く
texts = [chunk for chunk in chunks if chunk]
print(len(chunks), "個の断片のうち、空でないものは", len(texts), "個")
print(texts[:12])
print("つなげると:", "".join(texts))

61 個の断片のうち、空でないものは 21 個
['自然', '言語', '処理', '（NLP）とは、自然', '言語', 'をコンピュータが理解', '、生成', '、分析', 'できるようにする技術', '分野', 'です。これは、テキストデータから意味', 'を抽出し、感情']
つなげると: 自然言語処理（NLP）とは、自然言語をコンピュータが理解、生成、分析できるようにする技術分野です。これは、テキストデータから意味を抽出し、感情分析、機械翻訳、質問応答、言語モデルなど、さまざまなタスクを実行することで、情報の取得やコミュニケーションを円滑にする役割を果たします。


ストリーマーは、文字化けや単語の途中での切れ目を避けるため、空白・改行・漢字のところまで文字をためてから渡す。そのため、日本語では 1 文字ずつではなく、漢字で終わるところまでのまとまりで表示される（ひらがな・カタカナが続く間は、まとめて出る）。TextIteratorStreamer では、ためている間は空の文字列が届くので、上のように除いて使う。